# Notebook 4 — Multi-Agent Team: Bond Pricing and Duration

**Companion notebook to:** *From Exam Question to Autonomous Agent: Teaching Agentic AI Through Financial
Mathematics* (CAS Global Teaching Materials Innovation Challenge submission)

This notebook builds the two-specialist team described in **Section 5** of the case study: a Bond Specialist and
a Rate Risk Specialist, coordinated by a team leader in `TeamMode.coordinate`, answering a single request that
spans both of their tool sets.

**Before running:** get a free Gemini API key at https://aistudio.google.com/app/apikey and paste it into the
`GEMINI_API_KEY` cell below (or set it as a Colab secret named `GEMINI_API_KEY`).

## 0. Setup

This notebook runs unchanged in **Google Colab** or in a **local editor** (VS Code, PyCharm, JupyterLab) using the
`uv`-managed environment that ships alongside these notebooks (`pyproject.toml` + `uv.lock`). The cell below
detects which one it is running in and does the right thing automatically:

- **Colab**: installs the required packages directly into the Colab runtime (nothing to download beforehand).
- **Local**: assumes you already ran `uv sync` once in the project folder (see `README.md`), so the packages are
  already present in `.venv` — this cell skips installation and just confirms the imports work.

Either way, run this cell once per session.

In [ ]:
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "agno[google,opentelemetry,sqlite]", "google-genai",
            "openinference-instrumentation-agno", "python-dotenv",
        ],
        check=True,
    )
else:
    print(
        "Running outside Colab -- assuming packages were already installed via "
        "`uv sync` in this project's folder (see README.md). Skipping pip install."
    )

import agno, google.genai, dotenv  # noqa: F401 -- import check only
print("Environment ready.")

In [ ]:
import os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Option A (recommended): click the key icon in Colab's left sidebar, add a
    # secret named GEMINI_API_KEY, and this line picks it up automatically.
    try:
        from google.colab import userdata
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
    # Option B: no Colab secret found -- paste your key directly here instead.
    if not os.environ.get("GEMINI_API_KEY"):
        os.environ["GEMINI_API_KEY"] = "PASTE_YOUR_GEMINI_API_KEY_HERE"
else:
    # Local: reads GEMINI_API_KEY from a ".env" file in the project root.
    # Copy .env.example to .env and fill it in once -- see README.md.
    from dotenv import load_dotenv
    load_dotenv()

assert os.environ.get("GEMINI_API_KEY") and "PASTE_YOUR" not in os.environ["GEMINI_API_KEY"], (
    "GEMINI_API_KEY is not set. In Colab: add a secret named GEMINI_API_KEY, or "
    "paste your key directly into this cell. Locally: copy .env.example to .env "
    "in the project folder and add your key there."
)
print("GEMINI_API_KEY is set.")

## 1. Why split into specialists (Section 5)

A single agent's tool-selection reliability degrades as its tool set grows — more tools means more ways to pick
the wrong one. The fix demonstrated here: split a Financial Mathematics toolkit across narrowly scoped specialist
agents, coordinated by a team leader that decides how to split an incoming request and delegates each part to the
matching specialist by its stated role.

## 2. The Financial Mathematics and the math (Section 5)

The two specialists below wrap Financial Mathematics already covered in Notebooks 1-3, applied to a bond's cash
flows rather than a single lump sum or annuity. Briefly, before the code:

**Bond price.** A bond pays a level coupon `Fr` each period and a redemption value `C` at maturity. Its price is
just the present value of those cash flows -- coupons treated as an annuity-immediate, redemption treated as a
single future amount -- both calculations Notebook 1 and 2 already built:

$$P = Fr \cdot a_{\overline{n}|} + C v^n$$

**Macaulay duration.** A bond's price is a weighted sum of several cash flows landing at different times, so it is
natural to ask what the average timing of those payments is, weighted by how much each one contributes to the
price. That average, in periods, is Macaulay duration:

$$D = \frac{\sum_t t \cdot v^t \, CF_t}{\sum_t v^t \, CF_t} = \frac{\sum_t t \cdot v^t \, CF_t}{P}$$

A cash flow far in the future (like the redemption value) pulls this average later; a bond weighted toward early
coupons has a shorter duration.

**Modified duration.** Duration also turns out to measure something else useful: how sensitive the bond's price is
to a small change in yield. Rescaling Macaulay duration by the discount factor gives modified duration, a direct
measure of that price sensitivity:

$$D_{mod} = \frac{D}{1+i}$$

The functions below implement exactly these three relationships, each composed from tools Notebooks 1 and 2
already built and verified -- `bond_price` calls the same `annuity_immediate_pv` and `present_value` functions,
and `modified_duration` is a one-line rescaling of whatever `macaulay_duration` returns.

## 3. Python implementation: the underlying tools

In [ ]:
from agno.exceptions import RetryAgentRun


def present_value(future_value: float, annual_rate: float, years: float) -> float:
    """
    Compute the present value of a single future cash flow under
    compound interest.

    Args:
        future_value (float): The amount to be received or paid at a
            future date. Must be non-negative.
        annual_rate (float): The effective annual interest rate, as a
            decimal (e.g., 0.06 for 6%). Must be greater than -1.
        years (float): The number of years until the future date.
            Must be non-negative.

    Returns:
        float: The present value of the future amount.
    """
    if future_value < 0:
        raise RetryAgentRun("future_value must be non-negative. Re-check the request.")
    if years < 0:
        raise RetryAgentRun("years must be non-negative. Re-check the request.")
    if annual_rate <= -1:
        raise RetryAgentRun("annual_rate must be greater than -100%. Re-check the request.")
    return future_value * (1 + annual_rate) ** (-years)


def annuity_immediate_pv(payment: float, rate_per_period: float, n: int) -> float:
    """
    Compute the present value of a level annuity-immediate.

    Args:
        payment (float): The level payment amount per period. Must be
            non-negative.
        rate_per_period (float): The effective interest rate per
            payment period, as a decimal. Must be greater than -1.
        n (int): The number of payment periods. Must be positive.

    Returns:
        float: The present value of the annuity.
    """
    if payment < 0:
        raise RetryAgentRun("payment must be non-negative. Re-check the request.")
    if n <= 0:
        raise RetryAgentRun("n must be a positive number of periods. Re-check the request.")
    if rate_per_period <= -1:
        raise RetryAgentRun("rate_per_period must be greater than -100%. Re-check the request.")
    if rate_per_period == 0:
        return payment * n
    v = 1 / (1 + rate_per_period)
    a_n = (1 - v ** n) / rate_per_period
    return payment * a_n


def bond_price(coupon: float, redemption_value: float, yield_rate: float, n: int) -> float:
    """
    Compute a bond's price as the present value of its coupons plus
    its redemption value, valued immediately after a coupon payment
    (or at issue). Not valid between coupon dates.

    Args:
        coupon (float): The coupon payment per period (face value
            times coupon rate per period). Must be non-negative.
        redemption_value (float): The amount paid at maturity. Must
            be positive.
        yield_rate (float): The yield rate per period, as a decimal.
            Must be greater than -1.
        n (int): The number of coupon periods remaining. Must be
            positive.

    Returns:
        float: The bond's price.
    """
    if coupon < 0:
        raise RetryAgentRun("coupon must be non-negative. Re-check the request.")
    if redemption_value <= 0:
        raise RetryAgentRun("redemption_value must be positive. Re-check the request.")
    if n <= 0:
        raise RetryAgentRun("n must be a positive number of periods. Re-check the request.")
    if yield_rate <= -1:
        raise RetryAgentRun("yield_rate must be greater than -100%. Re-check the request.")
    return annuity_immediate_pv(coupon, yield_rate, n) + present_value(redemption_value, yield_rate, n)


def as_cash_flow_map(cash_flows: dict) -> dict:
    """
    Normalise a time-to-amount mapping supplied by a model. JSON
    object keys arrive as strings; this converts them to integers
    and the amounts to floats, and rejects an empty or malformed
    mapping.
    """
    if not cash_flows:
        raise RetryAgentRun("cash_flows cannot be empty. Re-check the request.")
    try:
        normalised = {int(t): float(cf) for t, cf in cash_flows.items()}
    except (TypeError, ValueError):
        raise RetryAgentRun(
            "cash_flows must map integer times to numeric amounts, "
            'e.g. {"1": 40, "2": 40, "3": 1040}. Re-check the request.'
        )
    if any(t <= 0 for t in normalised):
        raise RetryAgentRun("All cash flow times must be positive. Re-check the request.")
    return normalised


def macaulay_duration(cash_flows: dict, yield_rate: float) -> float:
    """
    Compute the Macaulay duration of a set of cash flows.

    Args:
        cash_flows (dict): A mapping of time (period number) to cash
            flow amount at that time, e.g. {"1": 40, "2": 40, "3":
            1040}. Must be non-empty, with all times positive.
        yield_rate (float): The discount rate, as a decimal per
            period. Must be greater than -1.

    Returns:
        float: The Macaulay duration, in periods.
    """
    cash_flows = as_cash_flow_map(cash_flows)
    if yield_rate <= -1:
        raise RetryAgentRun("yield_rate must be greater than -100%. Re-check the request.")
    v = 1 / (1 + yield_rate)
    pv_terms = {t: cf * v ** t for t, cf in cash_flows.items()}
    total_pv = sum(pv_terms.values())
    weighted = sum(t * pv for t, pv in pv_terms.items())
    return weighted / total_pv


def modified_duration(macaulay_duration_value: float, yield_rate: float) -> float:
    """
    Convert Macaulay duration to modified duration.

    Args:
        macaulay_duration_value (float): The Macaulay duration, in
            periods. Must be non-negative.
        yield_rate (float): The discount rate, as a decimal per
            period. Must be greater than -1.

    Returns:
        float: The modified duration.
    """
    if macaulay_duration_value < 0:
        raise RetryAgentRun("macaulay_duration_value must be non-negative. Re-check the request.")
    if yield_rate <= -1:
        raise RetryAgentRun("yield_rate must be greater than -100%. Re-check the request.")
    return macaulay_duration_value / (1 + yield_rate)


# Sanity check against the case study's own worked numbers (Section 5):
# a 5-year $1,000 bond, $40 annual coupons, yield 6% -> price $915.75, modified duration 4.35.
price = bond_price(40, 1000, 0.06, 5)
cash_flows = {t: 40 for t in range(1, 5)}
cash_flows[5] = 1040
d_mac = macaulay_duration(cash_flows, 0.06)
d_mod = modified_duration(d_mac, 0.06)
print(f"price: {price:.2f}   Macaulay duration: {d_mac:.2f}   modified duration: {d_mod:.2f}")

## 4. The specialists and the coordinating team (Section 5)

In [ ]:
from agno.agent import Agent
from agno.models.google import Gemini
from agno.team import Team, TeamMode

BASE_INSTRUCTIONS = [
    "Always use one of the available tools to perform any numerical "
    "calculation. Never state a computed numeric result unless it came "
    "directly from a tool call.",
    "If a request is ambiguous about a rate convention or cash flow "
    "detail, ask one brief clarifying question rather than guessing.",
]

bond_specialist = Agent(
    id="bond-specialist",
    name="Bond Specialist",
    role="Prices bonds given coupon, redemption value, yield rate, and term.",
    model=Gemini(id="gemini-3.5-flash", temperature=0.0),
    tools=[bond_price],
    instructions=BASE_INSTRUCTIONS,
    markdown=True,
)

rate_risk_specialist = Agent(
    id="rate-risk-specialist",
    name="Rate Risk Specialist",
    role="Computes Macaulay duration and modified duration for a set of cash flows.",
    model=Gemini(id="gemini-3.5-flash", temperature=0.0),
    tools=[macaulay_duration, modified_duration],
    instructions=BASE_INSTRUCTIONS + [
        "To build the cash_flows mapping for a bond, list each coupon "
        "at its period number and add the redemption value to the "
        "final coupon.",
    ],
    markdown=True,
)

financial_team = Team(
    name="Financial Mathematics Team",
    mode=TeamMode.coordinate,
    model=Gemini(id="gemini-3.5-flash", temperature=0.0),
    members=[bond_specialist, rate_risk_specialist],
    instructions=[
        "Delegate bond-pricing subtasks to the Bond Specialist and "
        "duration subtasks to the Rate Risk Specialist.",
        "Synthesise both members\' results into a single, coherent answer.",
    ],
)

financial_team.print_response(
    "Price a 5-year $1,000 bond with $40 annual coupons to yield 6%, "
    "and tell me its modified duration."
)

The team leader delegates the pricing sub-question to the Bond Specialist, which calls
`bond_price(40, 1000, 0.06, 5)`; it delegates the duration sub-question to the Rate Risk Specialist, which builds
the bond's cash-flow schedule and calls `macaulay_duration`, then `modified_duration`; the leader then combines
both specialists' answers — a price of **$915.75** and a modified duration of **4.35** — into one response. Note
what the leader itself never does: compute a number. Its entire job is deciding *who* computes, never computing.

## 5. Reliability evaluation: checking the delegation itself (Section 3.7)

In [ ]:
response = financial_team.run(
    "Price a 5-year $1,000 bond with $40 annual coupons to yield 6%, "
    "and tell me its modified duration."
)

delegated_to = {m.agent_id for m in (response.member_responses or [])}
assert delegated_to == {"bond-specialist", "rate-risk-specialist"}, (
    f"Expected delegation to both specialists, got: {delegated_to}"
)
print(f"Delegated to: {delegated_to} -- PASSED")

This checks the delegation decision itself, not just the final numbers — a team that sends both sub-questions to
one specialist, or involves a third, fails here even if the final numbers happen to be right (Section 3.7).

## 6. Two related delegation modes (Section 5)

`TeamMode.coordinate` (used above) splits one request across specialists and synthesizes their answers.
Two related modes handle simpler cases: `TeamMode.route` sends an entire request to one best-matched specialist
(no decomposition needed), and `TeamMode.broadcast` sends the same request to every specialist independently —
useful for getting multiple independent opinions rather than one synthesized answer. Try changing `mode=` above
and comparing the behavior on a single-topic question, such as just the bond price.

## Note: the full capstone

At full scale, the case study's architecture extends this same pattern to thirty-one Financial Mathematics tools
across six specialists (Figure 3). This notebook demonstrates the pattern at a scale small enough to run and inspect end to
end in a single session; the full six-specialist system is described in the case study but not reproduced here.